### Strathmore Institute of Mathematical Sciences | Master of Science in Data Science and Analytics
### DSA 8401 Applied Machine Learning | Capstone
#### 084045


#### Capstone Project: An End-to-End Behavioural Early-Warning Engine for Bank Account Dormancy and Disengagement Risk

**Dataset:** ai-lab/MBD-mini  
**Source:** Hugging Face 

### Imports

In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.compose  import ColumnTransformer

from sklearn.preprocessing import (RobustScaler, OneHotEncoder)

from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

print(f'Imports completed')

/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.11.0' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import (
/opt/anaconda3/lib/python3.13/site-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


Imports completed


In [3]:
# install hugging face package
%pip install -U datasets huggingface_hub pandas

Note: you may need to restart the kernel to use updated packages.


### Data Download

**Source:** [MBD-mini](https://huggingface.co/datasets/ai-lab/MBD-mini) (Sber AI Lab,
licence CC BY 4.0), a 10% client sample of the Multimodal Banking Dataset
([Mollaev et al., 2024](https://arxiv.org/abs/2409.17587)).

The repository holds four archives. Download detail, client_split, targets and ptls files:

| Archive | Contents|
|---|---|
| `detail.tar.gz` | Raw event tables: transactions, geo-activity, support dialogues|
| `client_split.tar.gz` | Client-to-fold assignment|
| `targets.tar.gz` | The authors' monthly product-purchase targets|
| `ptls.tar.gz` | Pre-aggregated sequences for one specific library|

In [4]:
# download MBD-mini file; target.tar.gz 
from huggingface_hub import hf_hub_download

targets_path = hf_hub_download(
    repo_id="ai-lab/MBD-mini",
    filename="targets.tar.gz",
    repo_type="dataset"
)

print(targets_path)

/Users/tintin/.cache/huggingface/hub/datasets--ai-lab--MBD-mini/snapshots/4b0d5b4c0b014e02af6c2e6b7b27851e6fb0d5c3/targets.tar.gz


In [5]:
# download  detail.tar.gz file
from huggingface_hub import hf_hub_download

detail_path = hf_hub_download(
    repo_id="ai-lab/MBD-mini",
    filename="detail.tar.gz",
    repo_type="dataset"
)

print(detail_path)

/Users/tintin/.cache/huggingface/hub/datasets--ai-lab--MBD-mini/snapshots/4b0d5b4c0b014e02af6c2e6b7b27851e6fb0d5c3/detail.tar.gz


In [6]:
# download  client_split.tar.gz file
from huggingface_hub import hf_hub_download

client_split_path = hf_hub_download(
    repo_id="ai-lab/MBD-mini",
    filename="client_split.tar.gz",
    repo_type="dataset"
)

print(client_split_path)

/Users/tintin/.cache/huggingface/hub/datasets--ai-lab--MBD-mini/snapshots/4b0d5b4c0b014e02af6c2e6b7b27851e6fb0d5c3/client_split.tar.gz


In [7]:
# download  ptls.tar.gz file
from huggingface_hub import hf_hub_download

ptls_path = hf_hub_download(
    repo_id="ai-lab/MBD-mini",
    filename="ptls.tar.gz",
    repo_type="dataset"
)

print(ptls_path)

/Users/tintin/.cache/huggingface/hub/datasets--ai-lab--MBD-mini/snapshots/4b0d5b4c0b014e02af6c2e6b7b27851e6fb0d5c3/ptls.tar.gz


In [8]:
import os

dataset_path = os.path.expanduser("~/Documents/School//School/MSc. DSA 4/DSA 8401 Applied Machine Learning/Capstone Project")

os.makedirs(dataset_path, exist_ok=True)

print(dataset_path)

/Users/tintin/Documents/School//School/MSc. DSA 4/DSA 8401 Applied Machine Learning/Capstone Project


In [9]:
dataset_path = (
    "/Users/tintin/.cache/huggingface/hub/"
    "datasets--ai-lab--MBD-mini/snapshots/"
    "4b0d5b4c0b014e02af6c2e6b7b27851e6fb0d5c3"
)

print(os.path.exists(dataset_path))

True


In [10]:
files = os.listdir(dataset_path)

for file in files:
    if file.endswith(".tar.gz"):
        file_path = os.path.join(dataset_path, file)

        size = os.path.getsize(file_path) / (1024**3)

        print(f"{file}: {size:.2f} GB")

targets.tar.gz: 0.01 GB
ptls.tar.gz: 1.30 GB
detail.tar.gz: 1.83 GB
client_split.tar.gz: 0.00 GB


In [11]:
# extracting all four archives
import tarfile

archives = [
    "targets.tar.gz",
    "client_split.tar.gz",
    "detail.tar.gz",
    "ptls.tar.gz"
]

for archive in archives:

    archive_path = os.path.join(dataset_path, archive)

    folder_name = archive.replace(".tar.gz", "")
    extract_path = os.path.join(dataset_path, folder_name)

    os.makedirs(extract_path, exist_ok=True)

    with tarfile.open(archive_path, "r:gz") as tar:
        tar.extractall(path=extract_path, filter="data")

    print(f"Successfully extracted: {archive}")

Successfully extracted: targets.tar.gz
Successfully extracted: client_split.tar.gz
Successfully extracted: detail.tar.gz
Successfully extracted: ptls.tar.gz


In [12]:
# inspection of extracted files
for folder in ["targets", "client_split", "detail", "ptls"]:

    folder_path = os.path.join(dataset_path, folder)

    print(f"\n{'='*50}")
    print(f"FOLDER: {folder}")
    print('='*50)

    for root, dirs, files in os.walk(folder_path):
        for file in files:
            file_path = os.path.join(root, file)

            size = os.path.getsize(file_path) / (1024**2)

            print(f"{file} | {size:.2f} MB")


FOLDER: targets
_SUCCESS | 0.00 MB
part-00083-44ca8b70-9d42-48f7-9dec-0a7a012af308.c000.snappy.parquet | 0.04 MB
part-00036-44ca8b70-9d42-48f7-9dec-0a7a012af308.c000.snappy.parquet | 0.03 MB
part-00042-44ca8b70-9d42-48f7-9dec-0a7a012af308.c000.snappy.parquet | 0.02 MB
part-00063-44ca8b70-9d42-48f7-9dec-0a7a012af308.c000.snappy.parquet | 0.03 MB
part-00017-44ca8b70-9d42-48f7-9dec-0a7a012af308.c000.snappy.parquet | 0.03 MB
part-00068-44ca8b70-9d42-48f7-9dec-0a7a012af308.c000.snappy.parquet | 0.02 MB
part-00021-44ca8b70-9d42-48f7-9dec-0a7a012af308.c000.snappy.parquet | 0.03 MB
part-00055-44ca8b70-9d42-48f7-9dec-0a7a012af308.c000.snappy.parquet | 0.02 MB
part-00074-44ca8b70-9d42-48f7-9dec-0a7a012af308.c000.snappy.parquet | 0.02 MB
part-00000-44ca8b70-9d42-48f7-9dec-0a7a012af308.c000.snappy.parquet | 0.02 MB
part-00049-44ca8b70-9d42-48f7-9dec-0a7a012af308.c000.snappy.parquet | 0.03 MB
part-00045-44ca8b70-9d42-48f7-9dec-0a7a012af308.c000.snappy.parquet | 0.02 MB
part-00031-44ca8b70-9d42-48f

In [14]:
# locate parquet files
from pathlib import Path

targets_dir = Path(dataset_path) / "targets"

parquet_files = list(targets_dir.rglob("*.parquet"))

print("Number of Parquet files:", len(parquet_files))

# Display the first 5 file paths
for file in parquet_files[:5]:
    print(file.relative_to(targets_dir))

Number of Parquet files: 435
targets/fold=0/part-00083-44ca8b70-9d42-48f7-9dec-0a7a012af308.c000.snappy.parquet
targets/fold=0/part-00036-44ca8b70-9d42-48f7-9dec-0a7a012af308.c000.snappy.parquet
targets/fold=0/part-00042-44ca8b70-9d42-48f7-9dec-0a7a012af308.c000.snappy.parquet
targets/fold=0/part-00063-44ca8b70-9d42-48f7-9dec-0a7a012af308.c000.snappy.parquet
targets/fold=0/part-00017-44ca8b70-9d42-48f7-9dec-0a7a012af308.c000.snappy.parquet


In [13]:
%pip install pyarrow

Note: you may need to restart the kernel to use updated packages.


In [15]:
import pyarrow.dataset as ds

In [ ]:
EXTRACTED = Path(dataset_path)

TABLES = {
    "client_split": EXTRACTED / "client_split" / "client_split",
    "trx": EXTRACTED / "detail" / "detail" / "trx",
    "geo": EXTRACTED / "detail" / "detail" / "geo",
    "dialog": EXTRACTED / "detail" / "detail" / "dialog",
    "targets": EXTRACTED / "targets" / "targets",
}

In [17]:
for name, path in TABLES.items():
    print(f"{name}: {'Found' if path.exists() else 'NOT FOUND'}")

client_split: Found
trx: Found
geo: Found
dialog: Found
targets: Found


In [18]:
datasets = {
    name: ds.dataset(path, format="parquet", partitioning="hive")
    for name, path in TABLES.items()
}

### Inventory: size, rows and columns

In [19]:
inventory = []

for name, dset in datasets.items():
    files = list(TABLES[name].rglob("*.parquet"))

    inventory.append({
        "table": name,
        "files": len(files),
        "size_MB": sum(f.stat().st_size for f in files) / 1e6,
        "rows": dset.count_rows(),
        "columns": len(dset.schema),
    })

inventory = pd.DataFrame(inventory).set_index("table")

inventory.style.format({
    "size_MB": "{:,.1f}",
    "rows": "{:,}",
})

,files,size_MB,rows,columns
table,,,,
client_split,1000,7.5,"100,224",2
trx,400,677.3,"38,764,930",15
geo,460,832.5,"66,295,724",6
dialog,415,673.9,"217,205",4
targets,435,11.5,"1,202,688",9


In [ ]:
# inspect tables schema; data types
for name, dset in datasets.items():
    print(f"\n{'=' * 60}")
    print(f"TABLE: {name}")
    print(f"{'=' * 60}")
    print(dset.schema)


TABLE: client_split
client_id: string
fold: int32
-- schema metadata --
org.apache.spark.version: '3.2.1'
org.apache.spark.sql.parquet.row.metadata: '{"type":"struct","fields":[{"' + 66

TABLE: trx
client_id: string
event_time: timestamp[ns]
amount: float
event_type: int32
event_subtype: int32
currency: int32
src_type11: int32
src_type12: int32
dst_type11: int32
dst_type12: int32
src_type21: int32
src_type22: int32
src_type31: int32
src_type32: int32
fold: int32
-- schema metadata --
org.apache.spark.version: '3.2.1'
org.apache.spark.sql.parquet.row.metadata: '{"type":"struct","fields":[{"' + 960

TABLE: geo
client_id: string
event_time: timestamp[ns]
geohash_4: int32
geohash_5: int32
geohash_6: int32
fold: int32
-- schema metadata --
org.apache.spark.version: '3.2.1'
org.apache.spark.sql.parquet.row.metadata: '{"type":"struct","fields":[{"' + 341

TABLE: dialog
client_id: string
event_time: timestamp[ns]
embedding: list<element: float>
  child 0, element: float
fold: int32
-- schema 

In [ ]:
# read data
for name, dset in datasets.items():
    print(f"\nTABLE: {name}")
    display(dset.head(5).to_pandas())


TABLE: client_split


,client_id,fold
0,091d957b9e571dc1ddb66017f32202d3a4f48583b6399a...,0
1,1171cec680efbb4fd268a9ade0e94512c7a6f1e08a98a6...,0
2,127042b9a315782dfba125603f5173144f2b1e9b86b0bd...,0
3,14a3fab5925a8c9271ab9e16c25fedb47122095476a741...,0
4,2421da099471e3c4cca875717d19c6853a80930a01fad1...,0



TABLE: trx


,client_id,event_time,amount,event_type,event_subtype,currency,src_type11,src_type12,dst_type11,dst_type12,src_type21,src_type22,src_type31,src_type32,fold
0,01a1aca1531e0d86b605759fedd36e1855f41a8a6a49b4...,2022-10-26 13:58:59.938408,1.021543e+06,16,18,11,22,47,780,21619,21490,83,364,58,0
1,01a1aca1531e0d86b605759fedd36e1855f41a8a6a49b4...,2021-02-09 07:43:20.639579,3.007656e+05,1,12,11,4,958,306,26766,21490,83,364,58,0
2,01a1aca1531e0d86b605759fedd36e1855f41a8a6a49b4...,2022-10-22 09:12:06.614108,6.061549e+05,16,18,11,22,47,780,21619,21490,83,364,58,0
3,01a1aca1531e0d86b605759fedd36e1855f41a8a6a49b4...,2021-02-12 15:12:20.660481,8.032591e+04,1,12,11,4,958,306,26766,21490,83,364,58,0
4,01a1aca1531e0d86b605759fedd36e1855f41a8a6a49b4...,2022-10-08 06:52:28.877461,1.222866e+05,16,18,11,22,47,780,21619,21490,83,364,58,0



TABLE: geo


,client_id,event_time,geohash_4,geohash_5,geohash_6,fold
0,397d52b11ce72699a5482e5397f63a03af1636c85e9e61...,2022-03-25 13:39:51.569288,41342,406579,1018528,0
1,397d52b11ce72699a5482e5397f63a03af1636c85e9e61...,2022-06-11 07:21:32.505485,41342,406579,1018528,0
2,397d52b11ce72699a5482e5397f63a03af1636c85e9e61...,2022-08-08 21:25:07.461323,41342,406579,1018528,0
3,397d52b11ce72699a5482e5397f63a03af1636c85e9e61...,2022-07-23 11:32:03.433401,41342,406579,1018528,0
4,397d52b11ce72699a5482e5397f63a03af1636c85e9e61...,2022-05-28 08:18:19.843885,41342,406579,1018528,0



TABLE: dialog


,client_id,event_time,embedding,fold
0,5226901c78c655767ac2a075e6edca6b410624f0c9d2fd...,2021-06-18 05:49:23.962741,"[0.393141, -0.036111, 0.311134, -0.197847, -0....",0
1,5226901c78c655767ac2a075e6edca6b410624f0c9d2fd...,2022-06-29 07:39:31.618769,"[0.302206, -0.381503, 0.472719, -0.284991, -0....",0
2,5226901c78c655767ac2a075e6edca6b410624f0c9d2fd...,2022-06-24 06:06:59.691367,"[0.570911, -0.395088, 0.546858, -0.413399, -0....",0
3,5226901c78c655767ac2a075e6edca6b410624f0c9d2fd...,2021-05-12 12:34:32.616245,"[0.232727, -0.284487, 0.584149, -0.524622, -0....",0
4,5226901c78c655767ac2a075e6edca6b410624f0c9d2fd...,2021-10-25 05:25:17.810891,"[0.316968, -0.149733, 0.426037, -0.395491, -0....",0



TABLE: targets


,client_id,mon,target_1,target_2,target_3,target_4,trans_count,diff_trans_date,fold
0,00bd0ecf3d5a33aa8756097967d07797dca4c98de9b61c...,2022-02-28,0,0,0,0,10,0,0
1,00bd0ecf3d5a33aa8756097967d07797dca4c98de9b61c...,2022-03-31,0,0,0,0,29,0,0
2,00bd0ecf3d5a33aa8756097967d07797dca4c98de9b61c...,2022-04-30,0,0,0,0,51,0,0
3,00bd0ecf3d5a33aa8756097967d07797dca4c98de9b61c...,2022-05-31,0,0,0,0,52,4,0
4,00bd0ecf3d5a33aa8756097967d07797dca4c98de9b61c...,2022-06-30,0,0,0,0,52,35,0


In [22]:
def peek(name: str, n: int = 5) -> pd.DataFrame:
    df = datasets[name].head(n).to_pandas()
    if "embedding" in df:  # 768 floats per row; show the length only
        df["embedding"] = df["embedding"].map(lambda v: f"<{len(v)} floats>")
    return df


print(datasets["trx"].schema.remove_metadata())
peek("trx")

client_id: string
event_time: timestamp[ns]
amount: float
event_type: int32
event_subtype: int32
currency: int32
src_type11: int32
src_type12: int32
dst_type11: int32
dst_type12: int32
src_type21: int32
src_type22: int32
src_type31: int32
src_type32: int32
fold: int32


,client_id,event_time,amount,event_type,event_subtype,currency,src_type11,src_type12,dst_type11,dst_type12,src_type21,src_type22,src_type31,src_type32,fold
0,01a1aca1531e0d86b605759fedd36e1855f41a8a6a49b4...,2022-10-26 13:58:59.938408,1.021543e+06,16,18,11,22,47,780,21619,21490,83,364,58,0
1,01a1aca1531e0d86b605759fedd36e1855f41a8a6a49b4...,2021-02-09 07:43:20.639579,3.007656e+05,1,12,11,4,958,306,26766,21490,83,364,58,0
2,01a1aca1531e0d86b605759fedd36e1855f41a8a6a49b4...,2022-10-22 09:12:06.614108,6.061549e+05,16,18,11,22,47,780,21619,21490,83,364,58,0
3,01a1aca1531e0d86b605759fedd36e1855f41a8a6a49b4...,2021-02-12 15:12:20.660481,8.032591e+04,1,12,11,4,958,306,26766,21490,83,364,58,0
4,01a1aca1531e0d86b605759fedd36e1855f41a8a6a49b4...,2022-10-08 06:52:28.877461,1.222866e+05,16,18,11,22,47,780,21619,21490,83,364,58,0


In [23]:
print(datasets["geo"].schema.remove_metadata())
peek("geo")

client_id: string
event_time: timestamp[ns]
geohash_4: int32
geohash_5: int32
geohash_6: int32
fold: int32


,client_id,event_time,geohash_4,geohash_5,geohash_6,fold
0,397d52b11ce72699a5482e5397f63a03af1636c85e9e61...,2022-03-25 13:39:51.569288,41342,406579,1018528,0
1,397d52b11ce72699a5482e5397f63a03af1636c85e9e61...,2022-06-11 07:21:32.505485,41342,406579,1018528,0
2,397d52b11ce72699a5482e5397f63a03af1636c85e9e61...,2022-08-08 21:25:07.461323,41342,406579,1018528,0
3,397d52b11ce72699a5482e5397f63a03af1636c85e9e61...,2022-07-23 11:32:03.433401,41342,406579,1018528,0
4,397d52b11ce72699a5482e5397f63a03af1636c85e9e61...,2022-05-28 08:18:19.843885,41342,406579,1018528,0


In [24]:
print(datasets["dialog"].schema.remove_metadata())
peek("dialog")

client_id: string
event_time: timestamp[ns]
embedding: list<element: float>
  child 0, element: float
fold: int32


,client_id,event_time,embedding,fold
0,5226901c78c655767ac2a075e6edca6b410624f0c9d2fd...,2021-06-18 05:49:23.962741,<768 floats>,0
1,5226901c78c655767ac2a075e6edca6b410624f0c9d2fd...,2022-06-29 07:39:31.618769,<768 floats>,0
2,5226901c78c655767ac2a075e6edca6b410624f0c9d2fd...,2022-06-24 06:06:59.691367,<768 floats>,0
3,5226901c78c655767ac2a075e6edca6b410624f0c9d2fd...,2021-05-12 12:34:32.616245,<768 floats>,0
4,5226901c78c655767ac2a075e6edca6b410624f0c9d2fd...,2021-10-25 05:25:17.810891,<768 floats>,0


In [25]:
print(datasets["targets"].schema.remove_metadata())
peek("targets")

client_id: string
mon: string
target_1: int32
target_2: int32
target_3: int32
target_4: int32
trans_count: int32
diff_trans_date: int32
fold: int32


,client_id,mon,target_1,target_2,target_3,target_4,trans_count,diff_trans_date,fold
0,00bd0ecf3d5a33aa8756097967d07797dca4c98de9b61c...,2022-02-28,0,0,0,0,10,0,0
1,00bd0ecf3d5a33aa8756097967d07797dca4c98de9b61c...,2022-03-31,0,0,0,0,29,0,0
2,00bd0ecf3d5a33aa8756097967d07797dca4c98de9b61c...,2022-04-30,0,0,0,0,51,0,0
3,00bd0ecf3d5a33aa8756097967d07797dca4c98de9b61c...,2022-05-31,0,0,0,0,52,4,0
4,00bd0ecf3d5a33aa8756097967d07797dca4c98de9b61c...,2022-06-30,0,0,0,0,52,35,0


In [26]:
print(datasets["client_split"].schema.remove_metadata())
peek("client_split")

client_id: string
fold: int32


,client_id,fold
0,091d957b9e571dc1ddb66017f32202d3a4f48583b6399a...,0
1,1171cec680efbb4fd268a9ade0e94512c7a6f1e08a98a6...,0
2,127042b9a315782dfba125603f5173144f2b1e9b86b0bd...,0
3,14a3fab5925a8c9271ab9e16c25fedb47122095476a741...,0
4,2421da099471e3c4cca875717d19c6853a80930a01fad1...,0


### Time span of each table

In [28]:
import pyarrow.compute as pc

spans = []
for name in ["trx", "geo", "dialog"]:
    lo = hi = None
    for batch in datasets[name].to_batches(columns=["event_time"], batch_size=2_000_000):
        mm = pc.min_max(batch.column(0))
        lo = mm["min"].as_py() if lo is None else min(lo, mm["min"].as_py())
        hi = mm["max"].as_py() if hi is None else max(hi, mm["max"].as_py())
    spans.append({"table": name, "first_event": lo, "last_event": hi})

months = datasets["targets"].to_table(columns=["mon"]).column(0)
spans.append({"table": "targets (mon)", "first_event": pc.min(months).as_py(), "last_event": pc.max(months).as_py()})
pd.DataFrame(spans).set_index("table")

,first_event,last_event
table,,
trx,2020-12-31 21:00:02.169545,2022-12-31 20:59:58.460193
geo,2021-12-11 19:27:07.507106,2022-12-31 20:59:58.725788
dialog,2021-02-10 09:36:07.022271,2022-12-31 20:25:11.730797
targets (mon),2022-02-28,2023-01-31


- **Transactions span 24 months** (January 2021 to December 2022). The dataset card
  says 12 months, which describes the `targets` table, not the event tables.
- **Geo data starts only in December 2021**, so geo features can exist only for scoring
  dates in 2022.

### Which clients appear in which table?

In [29]:
def unique_clients(dset) -> set:
    # Batch by batch: geo alone has 66M rows, and loading its client_id column
    # at once needs more memory than a typical laptop has.
    seen = set()
    for batch in dset.to_batches(columns=["client_id"], batch_size=1_000_000):
        seen.update(pc.unique(batch.column(0)).to_pylist())
    return seen


clients = {name: unique_clients(dset) for name, dset in datasets.items()}
all_clients = clients["client_split"]

coverage = pd.DataFrame({
    "clients": {n: len(c) for n, c in clients.items()},
    "share_of_all_clients": {n: len(c & all_clients) / len(all_clients) for n, c in clients.items()},
    "not_in_client_split": {n: len(c - all_clients) for n, c in clients.items()},
})
coverage.style.format({"clients": "{:,}", "share_of_all_clients": "{:.1%}", "not_in_client_split": "{:,}"})

,clients,share_of_all_clients,not_in_client_split
client_split,"100,224",100.0%,0
trx,"98,721",98.5%,0
geo,"72,573",72.4%,0
dialog,"46,320",46.2%,0
targets,"100,224",100.0%,0


In [30]:
no_trx = all_clients - clients["trx"]
print(f"Clients with no transactions at all: {len(no_trx):,}")
print(f"Clients with transactions but no geo: {len(clients['trx'] - clients['geo']):,}")
print(f"Clients with transactions but no dialogue: {len(clients['trx'] - clients['dialog']):,}")

Clients with no transactions at all: 1,503
Clients with transactions but no geo: 27,074
Clients with transactions but no dialogue: 52,715


### Summary

| | |
|---|---|
| Population | 100,224 corporate clients (`client_split`) |
| Core table | `trx`: 38.8M transactions, 24 months |
| Optional sources | `geo` (2022 only, ~73% of clients), `dialog` (~46% of clients) |
| Not used as a label | `targets` (product purchases, a different task) |

- Dormancy is not given; we will **derive it from `trx`** (no transaction for N days).
- Clients with no transactions can never "become" dormant, so they leave the population.
- Missing `geo` or `dialog` for a client is normal (a missing *modality*), not a data error.

### Data preparation and cleaning

The transaction table has 38.8 million rows. Before choosing a tool, estimate what pandas would need by 
loading one million rows and scaling up.

In [33]:
TRX = EXTRACTED / "detail" / "detail" / "trx"
GEO = EXTRACTED / "detail" / "detail" / "geo"
DIALOG = EXTRACTED / "detail" / "detail" / "dialog"
SPLIT = EXTRACTED / "client_split" / "client_split"

In [34]:
sample = ds.dataset(TRX, format="parquet", partitioning="hive").head(1_000_000).to_pandas()
per_million = sample.memory_usage(deep=True).sum() / 1e9
print(f"pandas memory for 1M rows: {per_million:.2f} GB")
print(f"estimated for all 38.8M rows: {per_million * 38.8:.1f} GB")
del sample

pandas memory for 1M rows: 0.17 GB
estimated for all 38.8M rows: 6.5 GB


The notebook will use **Polars lazy queries**: each query is planned first and then run
in streaming batches, so the full table never sits in memory. Small results (counts,
summaries) are converted to pandas for display and plotting. Cleaning also replaces the
string `client_id` with a compact integer `client_key`.

In [35]:
!pip install polars

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 911.5/911.5 kB 5.6 MB/s eta 0:00:00-:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 18.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [polars]2m1/2 [polars]


In [36]:
import polars as pl

In [37]:
trx = pl.scan_parquet(TRX / "**" / "*.parquet", hive_partitioning=True)
geo = pl.scan_parquet(GEO / "**" / "*.parquet", hive_partitioning=True)
dialog = pl.scan_parquet(DIALOG / "**" / "*.parquet", hive_partitioning=True)
split = pl.read_parquet(SPLIT / "**" / "*.parquet", hive_partitioning=True)


def run(lf: pl.LazyFrame) -> pl.DataFrame:
    # Execute a lazy query with the streaming engine
    return lf.collect(engine="streaming")


n_rows_raw = run(trx.select(pl.len())).item()
print(f"trx rows: {n_rows_raw:,}")
print(f"clients in client_split: {split.height:,}")

trx rows: 38,764,930
clients in client_split: 100,224


### Value Types

In [38]:
schema = trx.collect_schema()
pd.DataFrame({"dtype": [str(t) for t in schema.values()]}, index=list(schema.keys()))

,dtype
client_id,String
event_time,"Datetime(time_unit='ns', time_zone=None)"
amount,Float32
event_type,Int32
event_subtype,Int32
currency,Int32
src_type11,Int32
src_type12,Int32
dst_type11,Int32
dst_type12,Int32


In [39]:
# Client lookup table: one integer key per client
client_map = (
    split.sort("client_id")
    .with_row_index("client_key")
    .with_columns(pl.col("client_key").cast(pl.Int32))
    .select("client_key", "client_id", "fold")
)
print(f"{client_map.height:,} clients, keys 0 to {client_map['client_key'].max():,}")
print(f"duplicate client_ids in client_split: {split['client_id'].is_duplicated().sum()}")
client_map.head()

100,224 clients, keys 0 to 100,223
duplicate client_ids in client_split: 0


client_key,client_id,fold
i32,str,i64
0,"""000032cc38caee45fe031778bcf6af…",2
1,"""00005e39da5fb5968996cd49130281…",4
2,"""00012766278a7b25ddd3d6de329a44…",3
3,"""0001edbc5ab720f70a615ed9e8429d…",4
4,"""0002ddd816198d32474486d54f4bfe…",4


### Duplicates

In [40]:
n_unique_rows = run(trx.unique().select(pl.len())).item()
print(f"exact duplicate rows: {n_rows_raw - n_unique_rows:,}")

exact duplicate rows: 0


No two rows are identical across all columns. 

### Missing Values

In [41]:
nulls = run(trx.null_count()).to_pandas().T.rename(columns={0: "null_count"})
nulls["null_pct"] = nulls["null_count"] / n_rows_raw * 100
nulls.sort_values("null_count", ascending=False)

,null_count,null_pct
dst_type11,243824,0.628981
dst_type12,243824,0.628981
src_type11,217361,0.560716
src_type12,217361,0.560716
src_type31,78996,0.203782
src_type32,78996,0.203782
src_type21,42293,0.109101
src_type22,42293,0.109101
currency,118,0.000304
client_id,0,0.000000


In [42]:
# Which columns have missing values and are they missing on the same rows?
nullable = nulls.index[nulls["null_count"] > 0].tolist()
print("columns with missing values:", nullable)
together = run(trx.select(
    pl.any_horizontal([pl.col(c).is_null() for c in nullable]).sum().alias("any_null"),
    pl.all_horizontal([pl.col(c).is_null() for c in nullable]).sum().alias("all_null"),
)).to_pandas()
together

columns with missing values: ['currency', 'src_type11', 'src_type12', 'dst_type11', 'dst_type12', 'src_type21', 'src_type22', 'src_type31', 'src_type32']


,any_null,all_null
0,342181,0


In [43]:
# Which transaction types have the missing values?
by_type = run(
    trx.group_by("event_type").agg(
        pl.len().alias("rows"),
        pl.any_horizontal([pl.col(c).is_null() for c in nullable]).mean().alias("null_share"),
    ).sort("rows", descending=True)
).to_pandas()
by_type[by_type["null_share"] > 0].sort_values("null_share", ascending=False).head(10)

,event_type,rows,null_share
47,53,3648,0.734649
9,48,500698,0.366932
41,47,12153,0.263474
35,21,20046,0.037863
14,4,212882,0.024986
30,25,41042,0.013937
12,19,265519,0.011769
1,46,5594429,0.011582
28,16,42500,0.009435
16,6,185591,0.007145


**Result.** Nine columns have missing values, all of them category codes: the four
`src/dst_type1*` columns (0.56 to 0.63% of rows), the four `src_type2*/3*` columns
(0.11 to 0.20%), and `currency` (118 rows).

The gaps are concentrated in a few transaction types: type 53 is missing these
attributes on 73% of its rows, type 48 on 37%. That pattern suggests the attributes
do not apply to those transactions (for example, no counterparty), rather than
being lost at random.

**Decision.** Fill every missing code with a new code, **-1 = "unknown / not
applicable"**. All real codes are 1 or higher, so -1 cannot clash. No rows are dropped: every transaction still counts as activity,
which is what dormancy is about.

### Inconsistent data

#### Time

In [44]:
local = trx.with_columns((pl.col("event_time") + pl.duration(hours=3)).alias("t_local"))
bounds = run(local.select(pl.col("t_local").min().alias("first"), pl.col("t_local").max().alias("last")))
bounds

first,last
datetime[μs],datetime[μs]
2021-01-01 00:00:02.169545,2022-12-31 23:59:58.460193


Convert every timestamp to bank-local time (UTC+3). Without this, about
2,500 transactions would fall in "December 2020", and every daily and monthly count
near a midnight boundary would be slightly wrong.

#### Amounts

In [45]:
amount_checks = run(trx.select(
    (pl.col("amount") <= 0).sum().alias("zero_or_negative"),
    pl.col("amount").is_nan().sum().alias("nan"),
    (pl.col("amount") < 1).sum().alias("below_1"),
    pl.col("amount").min().alias("min"),
    pl.col("amount").max().alias("max"),
)).to_pandas()
amount_checks

,zero_or_negative,nan,below_1,min,max
0,0,0,131781,0.000003,1.370187e+12


No zero, negative or missing amounts. About 132,000 transactions (0.34%) are below 1
unit; these are plausibly fees or an effect of anonymised scaling and are kept. The
largest amount is about 1.4 trillion units, examined in section 6.

#### Currency

In [46]:
currency = run(
    trx.group_by("currency").agg(
        pl.len().alias("rows"),
        pl.col("client_id").n_unique().alias("clients"),
        pl.col("amount").median().alias("median_amount"),
    ).sort("rows", descending=True)
).to_pandas()
currency["share_of_rows"] = currency["rows"] / currency["rows"].sum()
currency

,currency,rows,clients,median_amount,share_of_rows
0,11.0,38724240,98721,1.802502e+04,9.989503e-01
1,3.0,20324,636,8.939498e+04,5.242883e-04
2,15.0,16041,716,4.059371e+05,4.138019e-04
3,4.0,3128,178,1.238560e+04,8.069149e-05
4,17.0,446,27,1.276568e+04,1.150524e-05
5,10.0,199,22,1.437745e+04,5.133506e-06
6,8.0,174,15,7.074836e+03,4.488593e-06
7,1.0,159,5,1.153735e+04,4.101645e-06
8,NaN,118,8,7.449037e+04,3.043988e-06
9,6.0,82,10,5.038098e+05,2.115314e-06


Currency code 11 carries 99.9% of transactions and is used by every
client. Twelve other codes share the
remaining 0.1%, and 118 rows have no currency.

#### Category codes: is each subtype inside one type?

In [47]:
nesting = run(
    trx.group_by("event_subtype").agg(pl.col("event_type").n_unique().alias("types_per_subtype"))
)
print(f"distinct event_type: {run(trx.select(pl.col('event_type').n_unique())).item()}")
print(f"distinct event_subtype: {nesting.height}")
print(f"subtypes used under more than one type: {(nesting['types_per_subtype'] > 1).sum()}")

distinct event_type: 55
distinct event_subtype: 60
subtypes used under more than one type: 57


Almost every subtype appears under several types, so `event_subtype` is **not** a
child of `event_type`; it is a separate dimension. Both are kept as independent codes.

#### Do clients and folds agree across tables?

In [48]:
trx_clients = run(trx.group_by("client_id").agg(pl.col("fold").first().alias("fold_trx")))
check = trx_clients.join(client_map, on="client_id", how="left")
print(f"clients in trx: {trx_clients.height:,}")
print(f"clients in trx but not in client_split: {check['client_key'].is_null().sum():,}")
print(f"clients whose fold differs between trx and client_split: {(check['fold_trx'] != check['fold']).sum():,}")
print(f"clients in client_split with no transactions: {client_map.height - trx_clients.height:,}")

clients in trx: 98,721
clients in trx but not in client_split: 0
clients whose fold differs between trx and client_split: 0
clients in client_split with no transactions: 1,503


Every transaction belongs to a known client, and folds agree between tables. The 1,503
clients with no transactions at all are kept in the client table but **excluded from
the modelling population**: a client who never transacted cannot "become" dormant.

### Data Distribution

In [49]:
q = [0, 0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99, 1]
amount_q = run(trx.select([pl.col("amount").quantile(x).alias(f"{x:.0%}") for x in q])).to_pandas().T
amount_q.columns = ["amount"]
amount_q

,amount
0%,3.021821e-06
1%,6.385846e+00
5%,7.725962e+01
25%,2.453869e+03
50%,1.803584e+04
75%,9.475955e+04
95%,9.048114e+05
99%,5.377146e+06
100%,1.370187e+12


### Data Cleaning

In [53]:
LOCAL_OFFSET = pl.duration(hours=3)          # UTC -> bank-local time (UTC+3)
UNKNOWN_CODE = -1                            # fill for missing category codes (all real codes are >= 1)
CODE_COLS = ["event_type", "event_subtype", "currency", "src_type11", "src_type12",
             "dst_type11", "dst_type12", "src_type21", "src_type22", "src_type31", "src_type32"]

trx_clean = (
    trx
    .join(client_map.lazy().select("client_id", "client_key"), on="client_id", how="inner")
    .with_columns(
        (pl.col("event_time") + LOCAL_OFFSET).alias("event_time_local"),
        pl.col(CODE_COLS).fill_null(UNKNOWN_CODE),
        pl.col("amount").log1p().cast(pl.Float32).alias("log_amount"),
    )
    .select(["client_key", "event_time_local", "amount", "log_amount", *CODE_COLS])
    .sort("client_key", "event_time_local")
)

OUT_TRX = "trx_clean.parquet"
trx_clean.sink_parquet(OUT_TRX, compression="zstd", row_group_size=1_000_000)

In [55]:
geo_checks = run(geo.select(
    pl.len().alias("rows"),
    pl.all_horizontal(pl.all().is_not_null()).not_().sum().alias("rows_with_nulls"),
))
geo_dups = run(geo.group_by("client_id", "event_time", "geohash_6").len().filter(pl.col("len") > 1).select(pl.col("len").sum() - pl.len())).item()
print(geo_checks)
print(f"geo exact duplicate rows (client, time, geohash_6): {geo_dups:,}")

geo_clean = (
    geo
    .join(client_map.lazy().select("client_id", "client_key"), on="client_id", how="inner")
    .with_columns((pl.col("event_time") + LOCAL_OFFSET).alias("event_time_local"))
    .select("client_key", "event_time_local", "geohash_4", "geohash_5", "geohash_6")
)
OUT_GEO =  "geo_clean.parquet"
geo_clean.sink_parquet(OUT_GEO, compression="zstd", row_group_size=1_000_000)
# print(f"wrote {OUT_GEO.name}: {OUT_GEO.stat().st_size / 1e6:,.0f} MB")

shape: (1, 2)
┌──────────┬─────────────────┐
│ rows     ┆ rows_with_nulls │
│ ---      ┆ ---             │
│ u32      ┆ u32             │
╞══════════╪═════════════════╡
│ 66295724 ┆ 0               │
└──────────┴─────────────────┘
geo exact duplicate rows (client, time, geohash_6): 0


In [57]:
dlg = dialog.collect()
emb_len = dlg["embedding"].list.len()
print(f"dialog rows: {dlg.height:,}")
print(f"embedding lengths: {sorted(emb_len.unique().to_list())}")
print(f"embeddings containing NaN: {dlg['embedding'].list.eval(pl.element().is_nan().any()).explode().sum():,}")
print(f"duplicate (client, time) rows: {dlg.select(pl.struct('client_id', 'event_time').is_duplicated().sum()).item():,}")

dialog_clean = (
    dlg.join(client_map.select("client_id", "client_key"), on="client_id", how="inner")
    .with_columns(
        (pl.col("event_time") + LOCAL_OFFSET).alias("event_time_local"),
        pl.col("embedding").cast(pl.Array(pl.Float32, 768)),
    )
    .select("client_key", "event_time_local", "embedding")
    .sort("client_key", "event_time_local")
)
OUT_DIALOG = "dialog_clean.parquet"
dialog_clean.write_parquet(OUT_DIALOG, compression="zstd")
# print(f"wrote {OUT_DIALOG.name}: {OUT_DIALOG.stat().st_size / 1e6:,.0f} MB")
del dlg, dialog_clean

dialog rows: 217,205
embedding lengths: [768]
embeddings containing NaN: 0
duplicate (client, time) rows: 0


In [58]:
trx_out = pl.scan_parquet(OUT_TRX)
activity = run(trx_out.group_by("client_key").agg(
    pl.len().alias("n_trx"),
    pl.col("event_time_local").min().alias("first_trx"),
    pl.col("event_time_local").max().alias("last_trx"),
))
geo_keys = run(pl.scan_parquet(OUT_GEO).select(pl.col("client_key").unique()))["client_key"]
dialog_keys = pl.read_parquet(OUT_DIALOG, columns=["client_key"])["client_key"].unique()

clients = (
    client_map
    .join(activity, on="client_key", how="left")
    .with_columns(
        pl.col("n_trx").fill_null(0),
        pl.col("client_key").is_in(geo_keys.implode()).alias("has_geo"),
        pl.col("client_key").is_in(dialog_keys.implode()).alias("has_dialog"),
    )
    .with_columns((pl.col("n_trx") > 0).alias("in_population"))
)
OUT_CLIENTS = "clients.parquet"
clients.write_parquet(OUT_CLIENTS)
# print(f"wrote {OUT_CLIENTS.name}: {clients.height:,} clients")
clients.head().to_pandas()

,client_key,client_id,fold,n_trx,first_trx,last_trx,has_geo,has_dialog,in_population
0,0,000032cc38caee45fe031778bcf6af05aa2aabe476acb8...,2,344,2021-09-16 13:51:50.458732,2022-12-31 15:32:37.174632,True,True,True
1,1,00005e39da5fb5968996cd49130281c8bd074a42e0e17a...,4,13,2022-08-10 11:15:44.567628,2022-11-28 01:21:15.006878,False,True,True
2,2,00012766278a7b25ddd3d6de329a44dec4c278d5807fae...,3,19,2022-09-04 15:59:34.096437,2022-10-12 12:56:11.991623,False,False,True
3,3,0001edbc5ab720f70a615ed9e8429df9b6c3f3c3999a51...,4,315,2021-01-27 10:02:38.926094,2022-12-26 09:05:13.817154,True,False,True
4,4,0002ddd816198d32474486d54f4bfe4f7b361119b5dc45...,4,641,2021-01-01 07:24:53.293748,2022-12-27 10:54:52.099492,True,False,True


### Validate the outputs

In [59]:
WINDOW_START, WINDOW_END = pl.datetime(2021, 1, 1), pl.datetime(2023, 1, 1)

validation = run(trx_out.select(
    pl.len().alias("rows"),
    pl.col("client_key").n_unique().alias("clients"),
    pl.sum_horizontal(pl.all().null_count()).alias("nulls"),
    ((pl.col("event_time_local") < WINDOW_START) | (pl.col("event_time_local") >= WINDOW_END)).sum().alias("outside_window"),
    pl.col("log_amount").is_nan().sum().alias("nan_log_amount"),
)).to_pandas()
validation

,rows,clients,nulls,outside_window,nan_log_amount
0,38764930,98721,0,0,0


In [60]:
row = validation.iloc[0]
assert row["rows"] == n_rows_raw, "row count changed"
assert row["nulls"] == 0, "nulls remain"
assert row["outside_window"] == 0, "events outside Jan 2021 - Dec 2022 local time"
assert row["nan_log_amount"] == 0
assert clients["in_population"].sum() == row["clients"]
print("All checks passed.")

All checks passed.


In [61]:
cleaning_log = pd.DataFrame([
    ["Raw transactions", f"{n_rows_raw:,}", "38.8M rows, 98,721 clients"],
    ["Exact duplicates removed", "0", "none found"],
    ["Same-timestamp rows", "kept", "1 pair, two genuine transactions"],
    ["Missing category codes", "filled with -1", "9 code columns, at most 0.63% per column"],
    ["Non-domestic currency", "kept, flagged by code", "0.1% of rows; excluded from amount features later"],
    ["Time zone", "UTC -> UTC+3", "all events now inside Jan 2021 - Dec 2022"],
    ["Amount outliers", "kept", "log_amount added instead of capping"],
    ["client_id", "-> client_key (int32)", "lookup in clients.parquet"],
    ["Clients with no transactions", f"{(~clients['in_population']).sum():,}", "flagged, excluded from population"],
], columns=["Step", "Result", "Note"])
cleaning_log

,Step,Result,Note
0,Raw transactions,"38,764,930","38.8M rows, 98,721 clients"
1,Exact duplicates removed,0,none found
2,Same-timestamp rows,kept,"1 pair, two genuine transactions"
3,Missing category codes,filled with -1,"9 code columns, at most 0.63% per column"
4,Non-domestic currency,"kept, flagged by code",0.1% of rows; excluded from amount features later
5,Time zone,UTC -> UTC+3,all events now inside Jan 2021 - Dec 2022
6,Amount outliers,kept,log_amount added instead of capping
7,client_id,-> client_key (int32),lookup in clients.parquet
8,Clients with no transactions,"1,503","flagged, excluded from population"


### Summary

| Check | Finding | Action |
|---|---|---|
| Value types | Codes stored as integers; `client_id` is a 64-char string | Integer `client_key` |
| Duplicates | None exact; one same-timestamp pair, genuinely different | Nothing removed |
| Missing values | 9 code columns, ≤ 0.63% each, tied to specific transaction types | Code -1 = not applicable |
| Time zone | Stored in UTC; bank runs on UTC+3 | Shifted to local time |
| Currency | 99.9% domestic (code 11) | Amount features on code 11 only |
| Amounts | No invalid values; heavy right skew | Kept; `log_amount` added |
| Clients | 1,503 with no transactions | Excluded from population |

**Outputs in `data/interim/`**

| File | Rows | Contents |
|---|---|---|
| `trx_clean.parquet` | 38,764,930 | Transactions in local time, sorted by client and time |
| `geo_clean.parquet` | 66,295,724 | Geo events in local time |
| `dialog_clean.parquet` | 217,205 | Support-dialogue embeddings (768 floats) |
| `clients.parquet` | 100,224 | One row per client: activity span, sources, population flag |

No rows were dropped from any table; every change is a type, time-zone or fill fix.